# 03 · Scaling laws in miniature: loss vs. parameters, data and compute

**Track:** Training LLMs & Frontier Research · **Runtime (CPU default):** ~1.5 minutes · **GPU:** optional (wider grid)

Why did labs spend hundreds of millions of dollars on single training runs before knowing if they would work? Because
of **scaling laws**: the empirical finding that language-model loss falls as a smooth power law in model size $N$,
data $D$ and compute $C$, so you can fit the curve on small runs and extrapolate. Kaplan et al. (OpenAI, 2020) put
this on the map; Hoffmann et al. (DeepMind, 2022, "Chinchilla") redid the analysis and concluded that models of the
day were **too big and under-trained**: for a fixed compute budget, parameters and tokens should grow roughly
equally. In this notebook we train seven tiny GPTs (from ~3K to ~800K parameters), fit both styles of scaling law,
and look honestly at how far a toy experiment can take this.

> **Source & license credit.** Model and training loop: [`research/src/tiny_gpt.py`](../src/tiny_gpt.py), adapted from
> **Karpathy's [nanoGPT](https://github.com/karpathy/nanoGPT) (MIT)**. The parametric fit follows the method described
> in Hoffmann et al. 2022 (Approach 3: Huber loss on log-loss, L-BFGS from a grid of initialisations); the code is
> original. Fitting uses [SciPy](https://scipy.org) (BSD-3-Clause). Data: public-domain Shakespeare excerpt. See
> [`research/THIRD_PARTY_NOTICES.md`](../THIRD_PARTY_NOTICES.md).

## Learning goals

1. Run a small, controlled **model-size × data sweep** and log loss at many token counts.
2. Fit a Kaplan-style power law $L(N) = E + (N_c/N)^{\alpha_N}$.
3. Fit the Chinchilla parametric form $L(N, D) = E + A/N^{\alpha} + B/D^{\beta}$.
4. Derive the **compute-optimal** allocation $N_{opt}(C) \propto C^{\beta/(\alpha+\beta)}$ under $C \approx 6ND$.
5. Say clearly why a toy fit like this one **should not** be trusted for extrapolation, and what would make it better.

## Background

* Kaplan, McCandlish, Henighan, Brown, Chess, Child, Gray, Radford, Wu, Amodei, *Scaling Laws for Neural Language
  Models*, OpenAI 2020. [arXiv:2001.08361](https://arxiv.org/abs/2001.08361)
  Reported $L(N) \approx (N_c/N)^{0.076}$, $L(D) \approx (D_c/D)^{0.095}$, $L(C_{min}) \propto C^{-0.050}$, and a
  compute-optimal $N_{opt} \propto C^{0.73}$ ("grow the model much faster than the data").
* Hoffmann, Borgeaud, Mensch, et al., *Training Compute-Optimal Large Language Models* ("Chinchilla"), DeepMind 2022.
  [arXiv:2203.15556](https://arxiv.org/abs/2203.15556)
  Fitted $L(N,D) = E + A/N^{\alpha} + B/D^{\beta}$ with $E{=}1.69, A{=}406.4, B{=}410.7, \alpha{=}0.34, \beta{=}0.28$,
  giving $N_{opt} \propto C^{0.46}$ (≈ 0.5 across their three approaches) → about **20 tokens per parameter**.
* Besiroglu et al., *Chinchilla Scaling: A replication attempt*, 2024. [arXiv:2404.10102](https://arxiv.org/abs/2404.10102)
  (re-fits the Chinchilla data; a good lesson in how fragile these fits are).
* Muennighoff et al., *Scaling Data-Constrained Language Models*, 2023. [arXiv:2305.16264](https://arxiv.org/abs/2305.16264)
  (repeating data for up to ~4 epochs is almost as good as fresh data; relevant because we repeat ours).
* Hägele et al., *Scaling Laws and Compute-Optimal Training Beyond Fixed Training Durations*, 2024.
  [arXiv:2405.18392](https://arxiv.org/abs/2405.18392) (constant LR + cooldown makes "one run, many D" sweeps cheap).

**Why the learning-rate schedule matters.** One of Chinchilla's key corrections to Kaplan: if every run uses a cosine
schedule tuned for the *longest* horizon, intermediate checkpoints are not representative of a run that was planned to
stop there, which biases the fit toward big models. We use a **constant LR after warmup**, so each checkpoint
approximates "a run of length D" (still imperfect, since it lacks the final cooldown).

In [ ]:
# --- Setup: make research/src importable (from research/, research/notebooks/, or a Colab clone) ---
import sys, pathlib
for _p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]:
    _src = next((c for c in (_p / "src", _p / "research" / "src") if (c / "lab_config.py").exists()), None)
    if _src:
        sys.path.insert(0, str(_src))
        break
else:
    raise RuntimeError("Could not find research/src -- run this notebook from inside the repo.")

from lab_config import CFG, pick, set_seed, heavy_cell, read_text, quiet_hf_logging

FULL_SIZE = CFG.full_size   # <- set True here (or export LAB_FULL_SIZE=1) for the full-size run
CFG.full_size = FULL_SIZE
set_seed()
print(CFG.summary())

In [ ]:
import math, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, minimize
from scipy.special import logsumexp
from tiny_gpt import GPT, GPTConfig, CharTokenizer, train_val_split, train

text = read_text("tiny_shakespeare.txt")
tok = CharTokenizer(text)
data = torch.tensor(tok.encode(text), dtype=torch.long)
train_data, val_data = train_val_split(data, 0.9)

# (n_layer, n_embd, n_head): widths and depths spanning ~2.5 orders of magnitude in N
GRID = pick(
    [(1, 16, 2), (2, 24, 2), (2, 32, 4), (2, 48, 4), (3, 64, 4), (4, 96, 4), (4, 128, 4)],
    [(1, 16, 2), (2, 32, 4), (2, 64, 4), (3, 96, 4), (4, 128, 4), (4, 192, 6), (6, 256, 8), (6, 384, 8)],
)
BLOCK = 64
BATCH = pick(16, 32)
MAX_STEPS = pick(800, 4000)
EVAL_AT = sorted({int(s) for s in np.geomspace(25, MAX_STEPS, 8)})
LR = 3e-3
print(f"tokens/step = {BATCH * BLOCK:,}; max tokens = {MAX_STEPS * BATCH * BLOCK:,} "
      f"(= {MAX_STEPS * BATCH * BLOCK / len(train_data):.1f} epochs of the train split)")
print("eval at steps:", EVAL_AT)

## 1 · The sweep

Same data order (seed), same optimiser and LR for every model; only the size changes. We record validation loss at
log-spaced token counts. Parameter count $N$ is **non-embedding** parameters, following Kaplan et al.

In [ ]:
runs = []
t_all = time.time()
for (n_layer, n_embd, n_head) in GRID:
    set_seed(CFG.seed)
    cfg = GPTConfig(vocab_size=tok.vocab_size, block_size=BLOCK, n_layer=n_layer, n_head=n_head, n_embd=n_embd)
    model = GPT(cfg).to(CFG.device)
    hist = train(model, train_data, val_data, max_steps=MAX_STEPS, batch_size=BATCH, lr=LR, warmup=30,
                 eval_every=10**9, eval_at=set(EVAL_AT), eval_iters=pick(20, 50), device=CFG.device,
                 seed=CFG.seed, schedule="constant", verbose=False)
    N = model.num_params()
    runs.append({"cfg": (n_layer, n_embd, n_head), "N": N, "hist": hist})
    print(f"L={n_layer} d={n_embd:4d}  N={N:>9,}  final val={hist['val_loss'][-1]:.3f}  ({hist['seconds']:.0f}s)")
print(f"sweep done in {time.time() - t_all:.0f}s")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
cmap = plt.get_cmap("viridis")
for i, r in enumerate(runs):
    h = r["hist"]
    toks = np.array(h["tokens"][1:]); vl = np.array(h["val_loss"][1:])  # drop step 0
    c = cmap(i / max(1, len(runs) - 1))
    ax[0].plot(toks, vl, "o-", color=c, label=f"N={r['N']:,}")
    ax[1].plot(6 * r["N"] * toks, vl, "-", color=c)
ax[0].set(xscale="log", yscale="log", xlabel="tokens seen D", ylabel="val loss (nats)", title="Loss vs data, per model size")
ax[1].set(xscale="log", yscale="log", xlabel="compute C ≈ 6·N·D (FLOPs)", ylabel="val loss", title="Loss vs compute")
ax[0].legend(fontsize=7); [a.grid(alpha=0.3, which="both") for a in ax]
plt.tight_layout(); plt.show()

The right panel is the picture behind compute-optimal training: each curve is one model size, and the **lower
envelope** tells you, for each compute budget, which model size you should have trained. Small models win at small
budgets and then plateau; bigger models start worse but keep improving.

## 2 · Kaplan-style fit: L(N) at the largest D

In [ ]:
N_arr = np.array([r["N"] for r in runs], dtype=float)
L_final = np.array([r["hist"]["val_loss"][-1] for r in runs])

def kaplan_N(N, E, logNc, alpha):
    return E + np.exp(alpha * (logNc - np.log(N)))

(E_k, logNc, alpha_N), _ = curve_fit(kaplan_N, N_arr, L_final, p0=[1.0, np.log(1e3), 0.3], maxfev=20000,
                                       bounds=([0, 0, 0.01], [L_final.min(), 40, 3]))
print(f"L(N) = {E_k:.3f} + (N_c / N)^{alpha_N:.3f},  N_c = {np.exp(logNc):.3g}")

# the pure power law Kaplan reported (no irreducible term), for comparison: log L = a - alpha * log N
slope, intercept = np.polyfit(np.log(N_arr), np.log(L_final), 1)
print(f"pure power-law fit: L ∝ N^{slope:.3f}   (Kaplan et al. 2020 report N^-0.076 at much larger scale)")

Ns = np.geomspace(N_arr.min() / 2, N_arr.max() * 4, 100)
plt.figure(figsize=(5.5, 3.6))
plt.plot(N_arr, L_final, "o", label="measured (final checkpoint)")
plt.plot(Ns, kaplan_N(Ns, E_k, logNc, alpha_N), "-", label=f"E + (Nc/N)^{alpha_N:.2f}")
plt.plot(Ns, np.exp(intercept) * Ns ** slope, "--", label=f"pure power law N^{slope:.2f}")
plt.xscale("log"); plt.yscale("log"); plt.xlabel("non-embedding parameters N"); plt.ylabel("val loss")
plt.legend(fontsize=8); plt.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()

## 3 · Chinchilla-style fit: L(N, D) on every checkpoint

We fit $\log L = \mathrm{LSE}(a - \alpha \log N,\; b - \beta \log D,\; e)$ (with $A = e^a$, $B = e^b$, $E = e^e$) by
minimising a Huber loss (δ = 10⁻³) with L-BFGS from a grid of initialisations, as described in Hoffmann et al. (2022),
Approach 3. Checkpoints in the first 100 steps are dropped because they are dominated by warmup.

In [ ]:
rows = []
for r in runs:
    h = r["hist"]
    for step, toks, vl in zip(h["step"], h["tokens"], h["val_loss"]):
        if step >= 100:
            rows.append((r["N"], toks, vl))
rows = np.array(rows, dtype=float)
logN, logD, logL = np.log(rows[:, 0]), np.log(rows[:, 1]), np.log(rows[:, 2])

def huber(x, delta=1e-3):
    a = np.abs(x)
    return np.where(a <= delta, 0.5 * x**2, delta * (a - 0.5 * delta))

def objective(p):
    a, b, e, alpha, beta = p
    pred = logsumexp(np.stack([a - alpha * logN, b - beta * logD, np.full_like(logN, e)]), axis=0)
    return huber(pred - logL).sum()

best = None
for a0 in (0, 5, 10):
    for b0 in (0, 5, 10):
        for e0 in (-1, 0, 0.5):
            for al0 in (0.2, 0.5, 1.0):
                for be0 in (0.2, 0.5, 1.0):
                    res = minimize(objective, [a0, b0, e0, al0, be0], method="L-BFGS-B",
                                   bounds=[(-10, 30), (-10, 30), (-3, 2), (0.01, 3), (0.01, 3)])
                    if best is None or res.fun < best.fun:
                        best = res
a, b, e, alpha, beta = best.x
A, B, E = np.exp(a), np.exp(b), np.exp(e)
print(f"fit on {len(rows)} checkpoints: L(N,D) = {E:.3f} + {A:.3g}/N^{alpha:.3f} + {B:.3g}/D^{beta:.3f}")
print(f"Chinchilla (2022):                L(N,D) = 1.69 + 406.4/N^0.34 + 410.7/D^0.28")

pred = np.exp(logsumexp(np.stack([a - alpha * logN, b - beta * logD, np.full_like(logN, e)]), axis=0))
print(f"mean |relative error| of the fit: {np.mean(np.abs(pred / rows[:, 2] - 1)) * 100:.2f}%")

## 4 · What allocation does our fit call compute-optimal?

Minimising $L(N, D)$ subject to $C = 6ND$ gives (Hoffmann et al. 2022, eq. 4)
$$N_{opt}(C) = G\left(\tfrac{C}{6}\right)^{\frac{\beta}{\alpha+\beta}},\quad D_{opt}(C) = G^{-1}\left(\tfrac{C}{6}\right)^{\frac{\alpha}{\alpha+\beta}},\quad G = \left(\tfrac{\alpha A}{\beta B}\right)^{\frac{1}{\alpha+\beta}}.$$

In [ ]:
a_exp = beta / (alpha + beta)
G = (alpha * A / (beta * B)) ** (1 / (alpha + beta))
print(f"Approach 3 (parametric):  N_opt ∝ C^{a_exp:.2f}   D_opt ∝ C^{1 - a_exp:.2f}")
print("   (Kaplan 2020: N_opt ∝ C^0.73;  Chinchilla 2022: N_opt ∝ C^0.46-0.50)\n")

# Only report allocations inside the compute range we actually measured -- extrapolation is where toy fits break.
C_meas = rows[:, 0] * rows[:, 1] * 6
for C in np.geomspace(C_meas.min() * 3, C_meas.max(), 4):
    N_opt = G * (C / 6) ** a_exp
    D_opt = (C / 6) / N_opt
    print(f"C={C:.1e}: N_opt≈{N_opt:>10,.0f} params, D_opt≈{D_opt:>12,.0f} tokens  ({D_opt / N_opt:,.0f} tokens/param)")

### Cross-check: Chinchilla's "Approach 1" (the empirical envelope)

Instead of trusting a 5-parameter formula, read the answer straight off the curves: for each compute budget, which of
our trained models reached the lowest loss? We interpolate each run's loss-vs-compute curve in log-log space, take the
best model at each budget, and fit a line to $\log N_{best}$ vs. $\log C$. With only seven discrete sizes this is
coarse, but it makes no assumption about the functional form. One trap: each run stops at $C = 6 N D_{max}$, so at
large budgets only the biggest models are still "in the race". We therefore only use budgets where at least three model
sizes have a checkpoint. (Chinchilla avoided this by training every size for several different horizons.)

In [ ]:
envelope = []
C_lo = max(6 * r["N"] * r["hist"]["tokens"][EVAL_AT.index(min(s for s in EVAL_AT if s >= 100)) + 1] for r in runs[:2])
for C in np.geomspace(C_lo, C_meas.max(), 14):
    cands = []
    for r in runs:
        c = 6 * r["N"] * np.array(r["hist"]["tokens"][1:], dtype=float)
        l = np.array(r["hist"]["val_loss"][1:])
        if c.min() <= C <= c.max():
            cands.append((float(np.exp(np.interp(np.log(C), np.log(c), np.log(l)))), r["N"]))
    if len(cands) >= 3:   # need several sizes competing at this budget, else the "winner" is just whoever reached it
        loss_best, N_best = min(cands)
        envelope.append((C, N_best, loss_best))
env = np.array(envelope, dtype=float)
slope1, icpt1 = np.polyfit(np.log(env[:, 0]), np.log(env[:, 1]), 1)
for C, N_best, lb in env[:: max(1, len(env) // 7)]:
    print(f"C={C:.1e}: best measured model N={int(N_best):>8,}  (loss {lb:.3f})")
print(f"\nApproach 1 (envelope, {len(env)} budgets): N_opt ∝ C^{slope1:.2f}   vs.   Approach 3 (parametric): C^{a_exp:.2f}")

Cs = np.geomspace(C_meas.min(), C_meas.max(), 50)
plt.figure(figsize=(5.5, 3.6))
for i, r in enumerate(runs):
    h = r["hist"]
    plt.plot(6 * r["N"] * np.array(h["tokens"][1:]), h["val_loss"][1:], color=cmap(i / max(1, len(runs) - 1)), alpha=0.7)
Nopt = G * (Cs / 6) ** a_exp
plt.plot(Cs, E + A / Nopt**alpha + B / ((Cs / 6) / Nopt) ** beta, "k--", label="Approach 3 fitted frontier")
plt.plot(env[:, 0], env[:, 2], "r.", label="Approach 1 envelope (best measured)")
plt.xscale("log"); plt.yscale("log"); plt.xlabel("C = 6ND"); plt.ylabel("val loss"); plt.legend(fontsize=8)
plt.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()

**Reading the two numbers.** Chinchilla's three approaches agreed with each other to within ~0.05 on the exponent.
If ours disagree by much more than that, neither number is a measurement: the envelope is quantised to seven model
sizes and a short compute range, and the parametric fit is extrapolating its $E$, $A$, $B$ from under-trained runs.
That disagreement is itself the most useful output of a toy scaling study: it tells you how much more range,
seeds and per-size tuning you would need before betting a big training run on the fit.

## 5 · How much should you trust this? (Not much, and here is why)

Treat the fitted exponents above as a **demonstration of the method, not a measurement**. Check the printed fit:
if the irreducible term $E$ collapses towards 0, or the two approaches disagree, or the implied tokens-per-parameter
is wildly far from Chinchilla's ~20, that is the data telling you the fit is under-determined, not a discovery.
In our CPU run every model is still improving steeply with $D$ (all are under-trained), so the $B/D^\beta$ term
dominates and the parametric fit pushes toward small models trained on many tokens. Concretely:

* **Tiny range.** Our $N$ spans about 2.5 orders of magnitude at the very bottom of the scale; Kaplan's spanned ~6 (768
  to 1.5B) and Chinchilla's runs went from 70M to 16B. Exponents fitted over a short range are highly uncertain, and
  small models are in a different regime (e.g. embedding- and width-limited).
* **One seed.** Seed-to-seed noise in tiny models is comparable to the gaps between neighbouring sizes. Error bars would
  need several seeds per point (see *Your turn*).
* **Repeated data.** The largest D here is a few epochs over 270 K characters. Muennighoff et al. (2023) show repetition
  is nearly free up to ~4 epochs, but beyond that $L(D)$ bends upward and the fit is no longer about "data" in the
  Chinchilla sense.
* **Un-tuned hyperparameters.** One LR for all sizes favours some models. Real scaling studies tune LR, batch size and
  schedule per scale (or use μP, Yang et al. 2022, [arXiv:2203.03466](https://arxiv.org/abs/2203.03466)).
* **No cooldown.** Constant-LR checkpoints overestimate the loss a properly scheduled run of that length would reach
  (Hägele et al. 2024). The bias depends on D, which leaks into $\beta$.
* **Character-level, single domain.** Exponents depend on the data distribution and tokenizer.

Even so, the qualitative picture usually survives at this scale: loss falls smoothly in $N$ and $D$, small models
saturate, and the lower envelope of loss-vs-compute is made of progressively larger models.

## 6 · (Optional, heavy) a wider sweep

`LAB_FULL_SIZE=1` switches to a grid up to ~10M parameters and 4,000 steps per model, about 1–1.5 orders of
magnitude more compute. It is **GPU-recommended** (roughly 10–20 minutes on a free Colab T4, much longer on CPU).
Everything above re-runs unchanged with the bigger grid; the cell below is just a reminder of what is enabled.

In [ ]:
if heavy_cell("Wider scaling grid (up to ~10M params, 4k steps each)", needs_gpu=False):
    print("FULL_SIZE grid in use:", GRID, "| steps:", MAX_STEPS)

## Your turn

1. **Error bars.** Re-run the sweep with 3 seeds (`set_seed(CFG.seed + k)`). Fit the Chinchilla form to each seed and
   report the spread of $\alpha$, $\beta$ and of the compute-optimal exponent. Is $\beta/(\alpha+\beta)$
   distinguishable from 0.5? From 0.73?
2. **Depth vs. width.** Kaplan et al. found loss depends mostly on $N$, weakly on shape. Hold $N$ roughly fixed and vary
   depth (1–6 layers). Does our toy agree?
3. **Embedding parameters.** Re-fit $L(N)$ using *total* instead of non-embedding parameters. Kaplan noted the trend
   becomes much less clean. Does it here?
4. **Tokens per parameter.** For the model closest to the fitted frontier at your largest compute, compute $D/N$.
   How far is it from Chinchilla's ~20?

## Beyond the paper — a mini research question

*Does the compute-optimal exponent depend on data repetition?* Shrink the training split to 25%, 50% and 100% of the
excerpt (so the same number of steps means 4× / 2× / 1× more epochs), re-run the sweep, and fit $L(N, D)$ each time.
Muennighoff et al. (2023) predict diminishing returns beyond ~4 epochs; can you see the point where "more tokens" stops
acting like more data, and how does that move $N_{opt}$? Bonus: add a 10–20% linear LR cooldown at the end of each
run (WSD schedule) and check whether $\beta$ changes, as Hägele et al. (2024) would suggest.

## References

* Kaplan, J. et al. (2020). *Scaling Laws for Neural Language Models.* [arXiv:2001.08361](https://arxiv.org/abs/2001.08361)
* Hoffmann, J. et al. (2022). *Training Compute-Optimal Large Language Models.* [arXiv:2203.15556](https://arxiv.org/abs/2203.15556)
* Besiroglu, T. et al. (2024). *Chinchilla Scaling: A replication attempt.* [arXiv:2404.10102](https://arxiv.org/abs/2404.10102)
* Muennighoff, N. et al. (2023). *Scaling Data-Constrained Language Models.* [arXiv:2305.16264](https://arxiv.org/abs/2305.16264)
* Hägele, A. et al. (2024). *Scaling Laws and Compute-Optimal Training Beyond Fixed Training Durations.* [arXiv:2405.18392](https://arxiv.org/abs/2405.18392)
* Yang, G. et al. (2022). *Tensor Programs V: Tuning Large Neural Networks via Zero-Shot Hyperparameter Transfer* (μP). [arXiv:2203.03466](https://arxiv.org/abs/2203.03466)
* Henighan, T. et al. (2020). *Scaling Laws for Autoregressive Generative Modeling.* [arXiv:2010.14701](https://arxiv.org/abs/2010.14701)